In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ── State Encoding & Model ────────────────────────────────────────────────────
def clues_to_board_state(clues: torch.Tensor, hidden: torch.Tensor, num_mines=16.0) -> torch.Tensor:
    B, _, H, W = clues.shape
    clue_vals = clues.squeeze(1).long().clamp(min=0)
    onehot = F.one_hot(clue_vals, num_classes=9).permute(0, 3, 1, 2).float()
    reveal_mask = 1.0 - hidden
    onehot = onehot * reveal_mask

    if isinstance(num_mines, torch.Tensor):
        mine_channel = num_mines.expand(B, 1, H, W) / (H * W)
    else:
        mine_channel = torch.full((B, 1, H, W), num_mines / (H * W), device=clues.device)

    return torch.cat([onehot, hidden, mine_channel], dim=1)  # (B, 11, H, W)

class MinesweeperEnergyModel(nn.Module):
    def __init__(self, hidden_dim: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(12, hidden_dim, kernel_size=5, padding=2), nn.GroupNorm(8, hidden_dim), nn.SiLU(),
            nn.Conv2d(hidden_dim, hidden_dim, kernel_size=5, padding=4, dilation=2), nn.GroupNorm(8, hidden_dim), nn.SiLU(),
            nn.Conv2d(hidden_dim, hidden_dim, kernel_size=5, padding=4, dilation=2), nn.GroupNorm(8, hidden_dim), nn.SiLU(),
            nn.Conv2d(hidden_dim, hidden_dim, kernel_size=5, padding=2), nn.GroupNorm(8, hidden_dim), nn.SiLU(),
            nn.Conv2d(hidden_dim, hidden_dim, kernel_size=5, padding=2), nn.GroupNorm(8, hidden_dim), nn.SiLU(),
            nn.Conv2d(hidden_dim, 1, kernel_size=5, padding=2),
        )

    def forward(self, board_state: torch.Tensor, candidate_mines: torch.Tensor) -> torch.Tensor:
        x = torch.cat([board_state, candidate_mines], dim=1)
        return self.net(x).sum(dim=[1, 2, 3])

# ── Energy Sampler ────────────────────────────────────────────────────────────
class EnergySampler:
    def __init__(self, model: nn.Module, num_steps: int = 30, alpha: float = 1.0, noise_std: float = 0.05):
        self.model = model
        self.num_steps = num_steps
        self.alpha = alpha
        self.noise_std = noise_std

    def _run_chain(self, board_state: torch.Tensor, steps: int) -> torch.Tensor:
        with torch.enable_grad():
            self.model.eval()
            for param in self.model.parameters(): param.requires_grad_(False)

            B, _, H, W = board_state.shape
            device = board_state.device
            revealed_mask = (board_state[:, 9:10, :, :] < 0.5)

            biases = torch.tensor([-1.40, 1.0, -2.0], device=device) # Prior, High-mine, Low-mine
            chosen_biases = biases[torch.randint(0, len(biases), (B, 1, 1, 1))]
            candidate_logits = torch.randn(B, 1, H, W, device=device) * 0.1 + chosen_biases

            candidate_logits = candidate_logits.masked_fill(revealed_mask, -10.0)
            candidate_logits.requires_grad_(True)

            for step_i in range(steps):
                candidate_probs = torch.sigmoid(candidate_logits)
                energy = self.model(board_state, candidate_probs)
                grad = torch.autograd.grad(energy.sum(), candidate_logits)[0]

                with torch.no_grad():
                    step_size = self.alpha / (1.0 + 0.1 * step_i)

                    if step_i < int(0.8 * steps):
                        noise_scale = (2 * step_size)**0.5 * self.noise_std
                        noise = noise_scale * torch.randn_like(candidate_logits)
                    else:
                        noise = 0.0

                    candidate_logits = candidate_logits - step_size * grad + noise
                    candidate_logits = candidate_logits.masked_fill(revealed_mask, -10.0) # Masking
                    candidate_logits = torch.clamp(candidate_logits, min=-10.0, max=10.0)

                candidate_logits.requires_grad_(True)

            for param in self.model.parameters(): param.requires_grad_(True)
            return torch.sigmoid(candidate_logits).detach()

    def sample_best_of_n(self, board_state: torch.Tensor, n: int = 16) -> torch.Tensor:
        expanded_board = board_state.repeat_interleave(n, dim=0)
        candidates = self._run_chain(expanded_board, self.num_steps)

        with torch.no_grad():
            energies = self.model(expanded_board, candidates).view(-1, n)
            best_idx = energies.argmin(dim=1)
            B, _, H, W = board_state.shape
            best = candidates.view(B, n, 1, H, W)[torch.arange(B, device=board_state.device), best_idx]
        return best

# ── Board generation ──────────────────────────────────────────────────────────
def _place_mines(height: int, width: int, num_mines: int, safe_r: int, safe_c: int):
    kernel = torch.ones(1, 1, 3, 3)
    kernel[0, 0, 1, 1] = 0
    forbidden = set()
    for dr in range(-1, 2):
        for dc in range(-1, 2):
            nr, nc = safe_r + dr, safe_c + dc
            if 0 <= nr < height and 0 <= nc < width:
                forbidden.add(nr * width + nc)
    available = [i for i in range(height * width) if i not in forbidden]
    chosen = torch.tensor(random.sample(available, num_mines))
    flat = torch.zeros(height * width)
    flat[chosen] = 1
    mines = flat.reshape(height, width)
    numbers = F.conv2d(mines.view(1, 1, height, width), kernel, padding=1).squeeze()
    return mines, numbers

def generate_board(height: int = 9, width: int = 9, num_mines: int = 16, max_clicks: int = 5):
    kernel = torch.ones(1, 1, 3, 3)
    kernel[0, 0, 1, 1] = 0
    first_r, first_c = random.randint(0, height - 1), random.randint(0, width - 1)
    mines, numbers = _place_mines(height, width, num_mines, first_r, first_c)
    hidden = torch.ones(height, width, dtype=torch.bool)

    def neighbors(r, c):
        for dr in [-1, 0, 1]:
            for dc in [-1, 0, 1]:
                if (dr or dc) and 0 <= r + dr < height and 0 <= c + dc < width:
                    yield r + dr, c + dc

    def click(r, c):
        stack = [(r, c)]
        while stack:
            r, c = stack.pop()
            if not hidden[r, c] or mines[r, c]: continue
            hidden[r, c] = False
            if numbers[r, c] == 0:
                stack += [(nr, nc) for nr, nc in neighbors(r, c) if hidden[nr, nc]]

    click(first_r, first_c)

    max_clicks = random.randint(1, 20)

    for _ in range(max_clicks - 1):
        adj_revealed = F.conv2d((~hidden).float().view(1, 1, height, width), kernel, padding=1).squeeze() > 0
        frontier = (hidden & adj_revealed).nonzero().tolist()
        safe = [(r, c) for r, c in frontier if not mines[r, c]]
        fallback = (hidden & (mines == 0)).nonzero().tolist()

        if safe: click(*random.choice(safe))
        elif fallback: click(*random.choice(fallback))
        else: break

    clues = torch.where(hidden, torch.tensor(-1.0), numbers)
    return clues.unsqueeze(0), mines.unsqueeze(0), hidden.float().unsqueeze(0)

def save_board_visual(clues: torch.Tensor, mines: torch.Tensor,
                      hidden: torch.Tensor, filepath: str):
    clues  = clues.squeeze().numpy()
    mines  = mines.squeeze().numpy()
    hidden = hidden.squeeze().numpy()

    H, W = clues.shape
    fig, ax = plt.subplots(figsize=(W * 0.6, H * 0.6))

    ax.set_xticks(np.arange(W + 1) - 0.5, minor=True)
    ax.set_yticks(np.arange(H + 1) - 0.5, minor=True)
    ax.grid(which="minor", color="black", linestyle="-", linewidth=2)
    ax.tick_params(which="minor", bottom=False, left=False)
    ax.set_xticks([]); ax.set_yticks([])

    colors = ["blue", "green", "red", "darkblue", "maroon", "cyan", "black", "gray"]

    for r in range(H):
        for c in range(W):
            if hidden[r, c]:
                face = "salmon" if mines[r, c] else "darkgray"
                ax.add_patch(plt.Rectangle((c - 0.5, r - 0.5), 1, 1,
                                           facecolor=face))
                if mines[r, c]:
                    ax.text(c, r, "M", va="center", ha="center",
                            color="darkred", fontweight="bold", fontsize=11)
            else:
                ax.add_patch(plt.Rectangle((c - 0.5, r - 0.5), 1, 1,
                                           facecolor="lightgray"))
                val = int(clues[r, c])
                if val > 0:
                    ax.text(c, r, str(val), va="center", ha="center",
                            color=colors[val - 1], fontweight="bold", fontsize=11)

    ax.set_xlim(-0.5, W - 0.5)
    ax.set_ylim(-0.5, H - 0.5)
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig(filepath, dpi=150)
    plt.close(fig)

# ── Dataset creation ──────────────────────────────────────────────────────────
def create_and_save_dataset(filepath: str = "minesweeper_train.pt",
                            num_samples: int = 100_000,
                            num_visuals:  int = 5,
                            height:       int = 9,
                            width:        int = 9,
                            num_mines:    int = 16,
                            max_clicks:   int = 5):
    os.makedirs("visuals", exist_ok=True)
    dataset = []

    print(f"Generating {num_samples:,} boards "
          f"({height}x{width}, {num_mines} mines)...")

    for i in range(num_samples):
        board_data = generate_board(height, width, num_mines, max_clicks)
        dataset.append(board_data)

        if i < num_visuals:
            clues, mines, hidden = board_data
            save_board_visual(clues, mines, hidden,
                              os.path.join("visuals", f"sample_{i+1}.png"))

        if (i + 1) % 5_000 == 0:
            print(f"  {i+1:>8,} / {num_samples:,}")

    torch.save(dataset, filepath)
    print(f"\nSaved {num_samples:,} boards → {filepath}")
    print(f"Saved {num_visuals} sample visuals → visuals/")

# ── Training Step ─────────────────────────────────────────────────────────────
class MinesweeperDataset(Dataset):
    def __init__(self, filepath: str):
        raw = torch.load(filepath, weights_only=True)
        self.clues  = torch.stack([r[0] for r in raw])   # (N, 1, H, W)
        self.mines  = torch.stack([r[1] for r in raw])   # (N, 1, H, W)
        self.hidden = torch.stack([r[2] for r in raw])   # (N, 1, H, W)
    def __len__(self):
        return len(self.clues)
    def __getitem__(self, idx):
        return self.clues[idx], self.mines[idx], self.hidden[idx]

def get_frontier_mask(hidden):
    revealed = 1.0 - hidden
    kernel = torch.ones((1, 1, 3, 3), device=hidden.device)
    neighbor_count = F.conv2d(revealed, kernel, padding=1)
    return (neighbor_count > 0) & (hidden > 0.5)

def ebm_train_step(model, clues, mines, hidden, *, alpha_base, alpha_factor,
                   noise_std=0.05, min_steps=3, max_steps=30,
                   cd_weight=0.1) -> torch.Tensor:   # ← new kwarg

    B, _, H, W = clues.shape
    device = clues.device

    true_mines_count = mines.sum(dim=[1, 2, 3]).view(B, 1, 1, 1)
    board_state = clues_to_board_state(clues, hidden, num_mines=true_mines_count)
    revealed_mask = (board_state[:, 9:10, :, :] < 0.5)

    energy_pos = model(board_state, mines.float())

    candidate_logits = (torch.randn(B, 1, H, W, device=device) * 0.1 - 1.40)
    candidate_logits = candidate_logits.masked_fill(revealed_mask, -10.0)
    candidate_logits.requires_grad_(True)

    steps = random.randint(min_steps, max_steps)
    for step_i in range(steps):
        candidate_probs = torch.sigmoid(candidate_logits)
        energy = model(board_state, candidate_probs)
        grad = torch.autograd.grad(energy.sum(), candidate_logits, create_graph=True)[0]

        log_factor = torch.empty(1).uniform_(-1.0, 1.0).item()
        alpha = alpha_base * (alpha_factor ** log_factor)
        step_size = alpha / (1.0 + 0.1 * step_i)

        noise_scale = (2 * step_size)**0.5 * noise_std
        noise = noise_scale * torch.randn_like(candidate_logits)

        candidate_logits = candidate_logits - step_size * grad + noise
        candidate_logits = candidate_logits.masked_fill(revealed_mask, -10.0)
        candidate_logits = torch.clamp(candidate_logits, -10.0, 10.0)

    final_probs = torch.sigmoid(candidate_logits)

    energy_neg = model(board_state, final_probs.detach())
    cd_loss = (energy_pos - energy_neg).mean()

    pixel_loss = F.binary_cross_entropy(final_probs, mines, reduction="none")
    pixel_loss = pixel_loss * hidden

    num_mines_t = mines.sum(dim=[1, 2, 3], keepdim=True)
    num_safe = (1.0 - mines).sum(dim=[1, 2, 3], keepdim=True)
    mine_weight = num_safe / num_mines_t.clamp(min=1)

    is_frontier_mine = mines.bool() & get_frontier_mask(hidden)
    weights = torch.where(is_frontier_mine, mine_weight, torch.ones_like(mine_weight))
    bce_loss = (pixel_loss * weights).sum() / hidden.sum().clamp(min=1)

    return bce_loss + cd_weight * cd_loss

# ── Sanity check ──────────────────────────────────────────────────────────────
def sanity_check(model: nn.Module, device: torch.device):
    model.eval()
    clues, mines, hidden = generate_board()
    clues  = clues.unsqueeze(0).to(device)
    mines  = mines.unsqueeze(0).to(device)
    hidden = hidden.unsqueeze(0).to(device)

    # 2nd Change applied to sanity check payload
    true_mines_count = mines.sum().view(1, 1, 1, 1)
    board  = clues_to_board_state(clues, hidden, num_mines=true_mines_count)

    with torch.no_grad():
        all_mines = hidden.clone()
        e_true    = model(board, mines).item()
        e_all     = model(board, all_mines).item()
    status = "GOOD" if e_true < e_all else "INVERTED — still learning"
    print(f"  [Sanity] E(true)={e_true:.4f}  E(all_mines)={e_all:.4f}  [{status}]")
    model.train()

# ── Main training loop ────────────────────────────────────────────────────────
def train(
    data_path:      str   = "minesweeper_train.pt",
    checkpoint_dir: str   = "checkpoints",
    *,
    use_checkpoint=True,
    epochs:         int   = 50,
    batch_size:     int   = 128,
    lr:             float = 3e-4,
    noise_std:      float = 0.05,
    min_steps:      int   = 3,
    max_steps:      int   = 30,
    log_every:      int   = 50,
    save_every:     int   = 5,
    device_str:     str   = "auto",
):
    device = (torch.device("cuda" if torch.cuda.is_available() else "cpu")
              if device_str == "auto" else torch.device(device_str))
    os.makedirs(checkpoint_dir, exist_ok=True)
    print(f"Training on {device}\n")
    dataset    = MinesweeperDataset(data_path)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True,
                            num_workers=4, pin_memory=True)
    model     = MinesweeperEnergyModel().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs * len(dataloader), eta_min=lr * 0.05
    )

    alpha_base, alpha_factor = 1.0, 2.0
    print()

    start_epoch = 1
    latest_ckpt = None
    if os.path.isdir(checkpoint_dir):
        ckpts = sorted([
            f for f in os.listdir(checkpoint_dir)
            if f.startswith("ebm_epoch") and f.endswith(".pt")
        ])
        if ckpts:
            latest_ckpt = os.path.join(checkpoint_dir, ckpts[-1])

    if latest_ckpt and use_checkpoint:
        print(f"Resuming from checkpoint: {latest_ckpt}")
        ckpt = torch.load(latest_ckpt, map_location=device, weights_only=False)
        model.load_state_dict(ckpt["model"])
        optimizer.load_state_dict(ckpt["optimizer"])
        scheduler.load_state_dict(ckpt["scheduler"])
        alpha_base   = ckpt.get("alpha_base",   alpha_base)
        alpha_factor = ckpt.get("alpha_factor", alpha_factor)
        start_epoch = ckpt["epoch"] + 1
        print(f"Resuming from epoch {start_epoch}\n")
    else:
        print("No checkpoint found, starting from scratch\n")

    for epoch in range(start_epoch, epochs + 1):
        model.train()
        epoch_loss = 0.0
        for step, (clues, mines, hidden) in enumerate(dataloader, 1):
            clues, mines, hidden = (t.to(device) for t in (clues, mines, hidden))
            optimizer.zero_grad()
            loss = ebm_train_step(
                model, clues, mines, hidden,
                alpha_base   = alpha_base,
                alpha_factor = alpha_factor,
                noise_std    = noise_std,
                min_steps    = min_steps,
                max_steps    = max_steps,
            )
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            epoch_loss += loss.item()
            if step % log_every == 0:
                avg = epoch_loss / step
                print(f"Epoch {epoch:>3} | {step:>5}/{len(dataloader)} "
                      f"| loss {loss.item():.4f} | avg {avg:.4f} "
                      f"| lr {scheduler.get_last_lr()[0]:.2e}")
        avg_epoch = epoch_loss / len(dataloader)
        print(f"\n── Epoch {epoch} complete — avg loss: {avg_epoch:.4f}")
        sanity_check(model, device)
        print()
        if epoch % save_every == 0:
            path = os.path.join(checkpoint_dir, f"ebm_epoch{epoch:03d}.pt")
            torch.save({
                "epoch":        epoch,
                "model":        model.state_dict(),
                "optimizer":    optimizer.state_dict(),
                "scheduler":    scheduler.state_dict(),
                "avg_loss":     avg_epoch,
                "alpha_base":   alpha_base,
                "alpha_factor": alpha_factor,
            }, path)
            print(f"Saved checkpoint → {path}\n")
    return model

In [ ]:
import random
import time
from dataclasses import dataclass, field
from typing import Optional

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import display, clear_output


# ── Game engine ───────────────────────────────────────────────────────────────
class MinesweeperGame:
    def __init__(self, height: int = 9, width: int = 9, num_mines: int = 16):
        self.height    = height
        self.width     = width
        self.num_mines = num_mines
        self.reset()

    def reset(self):
        self.mines          = torch.zeros(self.height, self.width)
        self.numbers        = torch.zeros(self.height, self.width)
        self.hidden         = torch.ones(self.height, self.width, dtype=torch.bool)
        self.exploded       = False
        self.move_count     = 0
        self._mines_placed  = False

    # ── mine placement ────────────────────────────────────────────────────────

    def _place_mines(self, safe_r: int, safe_c: int):
        kernel = torch.ones(1, 1, 3, 3)
        kernel[0, 0, 1, 1] = 0

        forbidden = set()
        for dr in range(-1, 2):
            for dc in range(-1, 2):
                nr, nc = safe_r + dr, safe_c + dc
                if 0 <= nr < self.height and 0 <= nc < self.width:
                    forbidden.add(nr * self.width + nc)

        available = [i for i in range(self.height * self.width) if i not in forbidden]
        chosen    = torch.tensor(random.sample(available, self.num_mines))

        flat          = torch.zeros(self.height * self.width)
        flat[chosen]  = 1
        self.mines    = flat.reshape(self.height, self.width)
        self.numbers  = F.conv2d(
            self.mines.view(1, 1, self.height, self.width), kernel, padding=1
        ).squeeze()
        self._mines_placed = True

    def _neighbors(self, r, c):
        for dr in [-1, 0, 1]:
            for dc in [-1, 0, 1]:
                if (dr or dc) and 0 <= r+dr < self.height and 0 <= c+dc < self.width:
                    yield r+dr, c+dc

    def _reveal(self, r, c):
        stack = [(r, c)]
        while stack:
            curr_r, curr_c = stack.pop()
            if not self.hidden[curr_r, curr_c] or self.mines[curr_r, curr_c]:
                continue
            self.hidden[curr_r, curr_c] = False
            if self.numbers[curr_r, curr_c] == 0:
                stack += [(nr, nc) for nr, nc in self._neighbors(curr_r, curr_c)
                          if self.hidden[nr, nc]]

    def click(self, r: int, c: int) -> str:
        if not self.hidden[r, c]:
            return "already"

        if not self._mines_placed:
            self._place_mines(r, c)

        self.move_count += 1

        if self.mines[r, c]:
            self.hidden[r, c] = False
            self.exploded = True
            return "mine"

        self._reveal(r, c)
        return "won" if self.is_won() else "safe"

    def is_won(self) -> bool:
        return bool((self.hidden == self.mines.bool()).all())

    def is_over(self) -> bool:
        return self.exploded or self.is_won()

    def get_clues_tensor(self) -> torch.Tensor:
        clues = torch.where(self.hidden, torch.tensor(-1.0), self.numbers)
        return clues.unsqueeze(0).unsqueeze(0)

    def get_hidden_tensor(self) -> torch.Tensor:
        return self.hidden.float().unsqueeze(0).unsqueeze(0)

    @property
    def num_hidden(self)       -> int: return int(self.hidden.sum())
    @property
    def num_hidden_mines(self) -> int: return int((self.hidden & self.mines.bool()).sum())


# ── Agent ─────────────────────────────────────────────────────────────────────
class EBMAgent:
    def __init__(self, model: MinesweeperEnergyModel,
                 sampler: EnergySampler, device: torch.device):
        self.model   = model
        self.sampler = sampler
        self.device  = device

    def get_mine_probs(self, game: MinesweeperGame) -> torch.Tensor:
        clues  = game.get_clues_tensor().to(self.device)
        hidden = game.get_hidden_tensor().to(self.device)
        board_state = clues_to_board_state(clues, hidden)

        # Sampler expects model to be in eval mode
        self.model.eval()
        mine_probs  = self.sampler.sample_best_of_n(board_state)    # (1, 1, H, W)
        return mine_probs.squeeze().cpu()                   # (H, W)

    def pick_cell(self, game: MinesweeperGame) -> tuple[int, int, torch.Tensor]:
        mine_probs  = self.get_mine_probs(game)
        hidden_mask = game.hidden.float()

        masked = mine_probs + (1.0 - hidden_mask) * 1e9
        flat   = masked.view(-1).argmin().item()
        return flat // game.width, flat % game.width, mine_probs


def load_agent_from_checkpoint(checkpoint_path: str, device: torch.device,
                               inference_steps: int = 15) -> EBMAgent:
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)

    model = MinesweeperEnergyModel().to(device)
    model.load_state_dict(checkpoint["model"])
    model.eval()

    alpha_base = checkpoint.get("alpha_base", 1.0)
    print(f"Loaded checkpoint '{checkpoint_path}' (Epoch {checkpoint.get('epoch', '?')})")
    print(f"Using calibrated alpha_base: {alpha_base:.4f} for {inference_steps} inference steps.")

    sampler = EnergySampler(
        model=model,
        num_steps=inference_steps,
        alpha=alpha_base,
        noise_std=0.0
    )

    return EBMAgent(model, sampler, device)

# ── Evaluation statistics ─────────────────────────────────────────────────────
@dataclass
class GameResult:
    won:               bool
    exploded:          bool
    moves:             int
    total_hidden:      int
    cells_revealed:    int
    mines_hit:         int
    prob_at_hit:       Optional[float] = None

@dataclass
class EvalStats:
    results: list[GameResult] = field(default_factory=list)

    @property
    def n(self):            return len(self.results)
    @property
    def win_rate(self):     return sum(r.won for r in self.results) / max(self.n, 1)
    @property
    def avg_moves(self):    return sum(r.moves for r in self.results) / max(self.n, 1)
    @property
    def avg_coverage(self):
        return sum(r.cells_revealed / max(r.total_hidden, 1)
                   for r in self.results) / max(self.n, 1)

    def summary(self) -> str:
        return (f"Games: {self.n} | "
                f"Win rate: {self.win_rate*100:.1f}% | "
                f"Avg moves: {self.avg_moves:.1f} | "
                f"Avg coverage: {self.avg_coverage*100:.1f}%")

# ── Evaluation loop ───────────────────────────────────────────────────────────
def evaluate(agent: EBMAgent, num_games: int = 200,
             height: int = 9, width: int = 9, num_mines: int = 16,
             verbose: bool = True) -> EvalStats:
    stats = EvalStats()
    game  = MinesweeperGame(height, width, num_mines)

    for idx in range(num_games):
        game.reset()
        initial_hidden = game.num_hidden
        mines_hit      = 0
        prob_at_hit    = None

        while not game.is_over():
            r, c, probs = agent.pick_cell(game)
            chosen_prob = probs[r, c].item()
            result = game.click(r, c)

            if result == "mine":
                mines_hit   = 1
                prob_at_hit = chosen_prob
                break

        cells_revealed = initial_hidden - game.num_hidden - mines_hit
        stats.results.append(GameResult(
            won            = game.is_won(),
            exploded       = game.exploded,
            moves          = game.move_count,
            total_hidden   = initial_hidden,
            cells_revealed = cells_revealed,
            mines_hit      = mines_hit,
            prob_at_hit    = prob_at_hit,
        ))

        if verbose and (idx + 1) % 10 == 0:
            print(f"  Game {idx+1:>4}/{num_games} — {stats.summary()}")

    return stats


# ── Visualisation ─────────────────────────────────────────────────────────────
def visualise_game(agent: EBMAgent, height: int = 9, width: int = 9,
                   num_mines: int = 16, max_steps: int = 480,
                   save_path: Optional[str] = None) -> bool:
    game        = MinesweeperGame(height, width, num_mines)
    clue_colors = ["blue", "green", "red", "darkblue",
                   "maroon", "cyan", "black", "gray"]

    def draw(ax_b, ax_h, title, probs=None, chosen=None, result=None):
        for ax in (ax_b, ax_h):
            ax.clear()
            ax.set_xticks([]); ax.set_yticks([])
            ax.set_xticks(np.arange(width  + 1) - 0.5, minor=True)
            ax.set_yticks(np.arange(height + 1) - 0.5, minor=True)
            ax.grid(which="minor", color="black", linewidth=1)

        ax_b.set_title(title, fontsize=8)
        ax_b.set_xlim(-0.5, width - 0.5)
        ax_b.set_ylim(-0.5, height - 0.5)
        ax_b.invert_yaxis()

        for r in range(height):
            for c in range(width):
                if game.hidden[r, c]:
                    face = "red" if (game.exploded and game.mines[r, c]) else "darkgray"
                    ax_b.add_patch(plt.Rectangle((c-.5, r-.5), 1, 1, facecolor=face))
                    if game.mines[r, c]:
                        ax_b.text(c, r, "M", ha="center", va="center",
                                  color="white", fontsize=8, fontweight="bold")
                else:
                    ax_b.add_patch(plt.Rectangle((c-.5, r-.5), 1, 1, facecolor="lightgray"))
                    val = int(game.numbers[r, c])
                    if val > 0:
                        ax_b.text(c, r, str(val), ha="center", va="center",
                                  color=clue_colors[val-1], fontweight="bold",
                                  fontsize=10)

        if chosen is not None:
            cr, cc  = chosen
            colour  = "red" if result == "mine" else "lime"
            ax_b.add_patch(mpatches.Rectangle(
                (cc-.5, cr-.5), 1, 1, facecolor="none", edgecolor=colour, linewidth=2.5
            ))

        if probs is not None:
            display_probs = np.where(game.hidden.numpy(), probs.numpy(), np.nan)
            ax_h.set_title("Mine probability", fontsize=8)
            ax_h.imshow(display_probs, vmin=0, vmax=1, cmap="RdYlGn_r",
                        origin="upper", aspect="equal")
            if chosen is not None:
                cr, cc = chosen
                ax_h.add_patch(mpatches.Rectangle(
                    (cc-.5, cr-.5), 1, 1, facecolor="none", edgecolor="blue", linewidth=2.5
                ))

    plt.ioff()
    fig, (ax_b, ax_h) = plt.subplots(1, 2, figsize=(11, 5))
    plt.colorbar(plt.cm.ScalarMappable(cmap="RdYlGn_r"),
                 ax=ax_h, fraction=0.046, pad=0.04, label="P(mine)")

    step = 0
    while not game.is_over() and step < max_steps:
        r, c, probs = agent.pick_cell(game)
        result = game.click(r, c)

        label = (f"Step {step+1} — clicked ({r},{c}) "
                 f"-> {'MINE' if result=='mine' else 'safe'}"
                 f"{'  WON!' if result=='won' else ''}")

        draw(ax_b, ax_h, label, probs=probs, chosen=(r, c), result=result)
        plt.tight_layout()

        if save_path:
            plt.savefig(f"{save_path}_step{step+1:02d}.png", dpi=100)

        step += 1

    outcome = "WON" if game.is_won() else "LOST"
    final_probs = agent.get_mine_probs(game)
    draw(ax_b, ax_h, f"Game over — {outcome} in {game.move_count} moves", probs=final_probs)
    plt.tight_layout()
    if save_path:
        plt.savefig(f"{save_path}_final.png", dpi=100)

    clear_output(wait=True)
    display(fig)

    plt.close(fig)
    plt.ion()

    return game.is_won()

# ── System 2 benchmark ────────────────────────────────────────────────────────
def benchmark(model: MinesweeperEnergyModel, alpha_base: float,
              device: torch.device, num_games: int = 300,
              steps_list: list[int] = [1, 5, 15, 30, 50]):
    results = {}

    for n_steps in steps_list:
        print(f"\n── {n_steps} inference steps ──")
        sampler = EnergySampler(model, num_steps=n_steps, alpha=alpha_base)
        agent   = EBMAgent(model, sampler, device)
        stats   = evaluate(agent, num_games=num_games, verbose=True)
        results[n_steps] = stats
        print(f"   {stats.summary()}")

    win_rates = [results[s].win_rate * 100  for s in steps_list]
    coverages = [results[s].avg_coverage * 100 for s in steps_list]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

    ax1.plot(steps_list, win_rates, "o-", color="steelblue", linewidth=2, markersize=8)
    ax1.set_xlabel("Inference steps (NFEs)"); ax1.set_ylabel("Win rate (%)")
    ax1.set_title("System 2 Thinking — Win Rate vs Compute")
    ax1.grid(True, alpha=0.3); ax1.set_xticks(steps_list)

    ax2.plot(steps_list, coverages, "o-", color="seagreen", linewidth=2, markersize=8)
    ax2.set_xlabel("Inference steps (NFEs)"); ax2.set_ylabel("Board coverage (%)")
    ax2.set_title("System 2 Thinking — Coverage vs Compute")
    ax2.grid(True, alpha=0.3); ax2.set_xticks(steps_list)

    plt.suptitle(f"EBM Minesweeper — {num_games} games each", fontsize=12)
    plt.tight_layout()
    plt.savefig("system2_scaling.png", dpi=150)
    print("\nSaved → system2_scaling.png")
    plt.show()
    return results

In [ ]:
create_and_save_dataset("minesweeper_train.pt", num_samples=100_000)

In [ ]:
train("minesweeper_train.pt", epochs=50, use_checkpoint=False)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available()  else "cpu")

ckpt    = torch.load("checkpoints/ebm_epoch050.pt")
model   = MinesweeperEnergyModel().to(device)
model.load_state_dict(ckpt["model"])

agent = load_agent_from_checkpoint("checkpoints/ebm_epoch050        .pt", device, inference_steps=15)

evaluate(agent, num_games=200)

In [ ]:
print("starting game")
visualise_game(agent)

In [ ]:
benchmark(
    model=model,
    alpha_base=ckpt["alpha_base"],
    device=device,
)